# 01 · Teacher: label the biographies on Colab

Run with **Runtime → Change runtime type → T4 GPU**.
Produces `labels.jsonl` (same format as `label.py` with Bedrock) and uploads it to S3.
If Colab disconnects, run cells 1, 2 and 4 again: labeling resumes where it stopped.

## 1. Set up the environment

In [ ]:
# Install dependencies (awscli: Colab does not ship the AWS CLI)
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets boto3 pydantic awscli

import os
from google.colab import drive, userdata

# Colab secrets (key icon, on the left): AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, DATA_BUCKET
for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "DATA_BUCKET"):
    os.environ[k] = userdata.get(k).strip()
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

drive.mount("/content/drive")  # anything slow goes to Drive: survives disconnects
DRIVE = "/content/drive/MyDrive/wiki-bio-distill"
os.makedirs(DRIVE, exist_ok=True)

# Repository: public, or private with a GITHUB_TOKEN secret
try:
    token = userdata.get("GITHUB_TOKEN")
    url = f"https://{token}@github.com/pinthoz/wiki-bio-distill.git"
except Exception:
    url = "https://github.com/pinthoz/wiki-bio-distill.git"
if not os.path.exists("/content/repo"):
    !git clone -q {url} /content/repo
%cd /content/repo
!git pull -q
!mkdir -p data preds
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Fetch the texts and link the output file to Drive

In [ ]:
!aws s3 cp s3://$DATA_BUCKET/v1/raw.jsonl data/
!wc -l data/raw.jsonl

# labels.jsonl lives on Drive; data/labels.jsonl is just a link to it
!touch {DRIVE}/labels.jsonl
!ln -sf {DRIVE}/labels.jsonl data/labels.jsonl

## 3. Trial run with 50 examples

Check the error rate, the speed and a few labels before launching the full run.

In [ ]:
!python label_local.py --model Qwen/Qwen2.5-7B-Instruct --limit 50

In [ ]:
import json
rows = {r["id"]: r for r in map(json.loads, open("data/raw.jsonl"))}
labels = [json.loads(l) for l in open("data/labels.jsonl")]
print(f"{len(labels)} labeled, {sum('error' in l for l in labels)} with errors\n")
for l in labels[:5]:
    print(rows[l["id"]]["text"][:200], "…")
    print(json.dumps(l.get("label") or l, ensure_ascii=False, indent=2), "\n")

## 4. Label the rest

For 3000 biographies, expect one to two hours. The script shows the time left.

In [ ]:
!python label_local.py --model Qwen/Qwen2.5-7B-Instruct

## 5. Upload to S3

In [ ]:
!grep -c '"error"' data/labels.jsonl || true
!aws s3 cp data/labels.jsonl s3://$DATA_BUCKET/v1/labels.jsonl

After this, on your laptop: `aws s3 cp s3://$DATA_BUCKET/v1/labels.jsonl data/`, `python review.py` and `python split.py` (step 4 of the guide), then upload `train`, `val` and `test` to S3.